In [ ]:
%load_ext autoreload
%autoreload 2

### get clumped results

from pathlib import Path
import pandas as pd
import re
import numpy as np
import json

from gwas_postproc import read_pairs, check_all_runs, summarize_missing

OUTROOT = Path("<OUTROOT>")
STEP2_ROOT = OUTROOT / "step2"
PAIRS = OUTROOT / "pairs_all.tsv"

pairs_df = read_pairs(PAIRS)

check_df = check_all_runs(pairs_df, STEP2_ROOT)
per_run, missing_rows = summarize_missing(check_df)

pairs_df = pd.read_csv("pairs_all.tsv", sep="\t")

dfs = []
for run_id in pairs_df["run_id"].astype(str):
    for f in (STEP2_ROOT / run_id).glob("chr*/*clump*.clumps"):
        df = pd.read_csv(f, sep=r"\s+", engine="python").rename(columns={"#CHROM": "CHROM"})
        df["run_id"] = run_id
        df["clumps_path"] = str(f)
        dfs.append(df)

clumps = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
clumps = clumps.merge(pairs_df, on="run_id", how="left")

clumps.run_id.value_counts().to_csv("regenie_clumps.tsv")

clumps = clumps[clumps["NEG_LOG10_P"] > 7.30103]

### read in CI-GWAS

cigwas = pd.read_csv(
    "<CI_GWAS_FULL_RESULTS_TSV>",
    sep = "\t")

cigwas = cigwas[cigwas.setup.isin([
    "sbp_pre_post_1to60_no_cvd",
    "dbp_pre_post_1to60_no_cvd",
    "sbp_pre_post_1to60_age5_with_statins_no_cvd",
    "dbp_pre_post_1to60_age5_with_statins_no_cvd"
])]

cigwas = cigwas[cigwas["p_fdr"] < 0.05]

ci_to_gwas_dict_pooled = {
    "SBP_pre_ADJ": "pooled__SBP_pre__base",
    "DBP_pre_ADJ": "pooled__DBP_pre__base",
    "SBP_post_ADJ": "pooled__SBP_post__base+classes+pre",
    "DBP_post_ADJ": "pooled__DBP_post__base+classes+pre",
    "angiotensin_receptor_blocker": "pooled__CLASS_angiotensin_receptor_blocker__base+others+SBP_pre",
    "statin": "pooled__CLASS_statin__base+others+SBP_pre",
    "beta_blocker": "pooled__CLASS_beta_blocker__base+others+SBP_pre",
    "ACE_inhibitor": "pooled__CLASS_ACE_inhibitor__base+others+SBP_pre",
    "diuretic": "pooled__CLASS_diuretic__base+others+SBP_pre",
    "calcium_channel_blocker":  "pooled__CLASS_calcium_channel_blocker__base+others+SBP_pre"
}

def build_ci_to_gwas_dict_age5():
    d = {}

    for k in range(1, 6):
        d[f"SBP_pre_{k}_ADJ"]  = f"age5g{k}__SBP_pre__base"
        d[f"SBP_post_{k}_ADJ"] = f"age5g{k}__SBP_post__base+classes+pre"
        d[f"DBP_pre_{k}_ADJ"]  = f"age5g{k}__DBP_pre__base"
        d[f"DBP_post_{k}_ADJ"] = f"age5g{k}__DBP_post__base+classes+pre"

    drugs = [
        "angiotensin_receptor_blocker",
        "statin",
        "beta_blocker",
        "ACE_inhibitor",
        "diuretic",
        "calcium_channel_blocker"
    ]
    for k in range(1, 6):
        for drug in drugs:
            d[f"{drug}_{k}"] = f"age5g{k}__CLASS_{drug}__base+others+SBP_pre"

    return d

ci_to_gwas_dict_age5 = build_ci_to_gwas_dict_age5()
ci_to_gwas = {**ci_to_gwas_dict_pooled, **ci_to_gwas_dict_age5}

_cache = {}

def step2_file(run_id, chrom):
    d = STEP2_ROOT / run_id / f"chr{chrom}"
    f = sorted(d.glob(f"{run_id}_chr{chrom}_step2*.regenie"))
    return f[0]

def load_rg(run_id, chrom):
    key = (run_id, chrom)
    if key not in _cache:
        df = pd.read_csv(step2_file(run_id, chrom), sep=r"\s+")
        _cache[key] = df[["ID", "LOG10P"]]
    return _cache[key]

def rg_p(run_id, chrom, rsid):
    run_id = str(run_id)
    df = load_rg(run_id, chrom)
    s = df.loc[df["ID"].eq(rsid), "LOG10P"]
    return float(10 ** (-s.iloc[0])) if not s.empty else np.nan

ci_df = cigwas.copy()
ci_df["regenie_run_id"] = ci_df["phenotype"].map(ci_to_gwas)
ci_df["regenie_p"] = [
    rg_p(r, int(c), v)
    for r, c, v in zip(ci_df["regenie_run_id"], ci_df["chr"], ci_df["rsID"])
]

ci_df.to_csv("comparison_with_regenie_p_values.tsv", sep="\t")

ci_df = pd.read_csv("comparison_with_regenie_p_values.tsv", sep="\t")

keep_rows = {
    ("sbp_pre_post_1to60_age5_with_statins_no_cvd", "angiotensin_receptor_blocker_1", "SLC35F2"),
    ("sbp_pre_post_1to60_age5_with_statins_no_cvd", "angiotensin_receptor_blocker_1", "PKD1"),

    ("sbp_pre_post_1to60_no_cvd", "statin", "rs11591147"),
    ("sbp_pre_post_1to60_no_cvd", "statin", "rs56393506"),
    ("sbp_pre_post_1to60_no_cvd", "statin", "rs72658867"),
    ("sbp_pre_post_1to60_no_cvd", "statin", "rs7412"),
}

k = list(zip(ci_df["setup"], ci_df["phenotype"], ci_df["rsID"]))
is_special_pheno = ci_df["phenotype"].isin(["angiotensin_receptor_blocker_1", "statin"])
keep_mask = (~is_special_pheno) | pd.Series([t in keep_rows for t in k], index=ci_df.index)

ci_df = ci_df.loc[keep_mask].copy()

### Interaction p-values (no significant SNPs genome-wide)

# We are interested in the interaction results for drug-taking variants and for post-SNPs

ci_to_gwas = {k: v for k, v in ci_to_gwas.items() if "pre" not in k}

INTX_TEXT = """
intx__DBP_delta__INT_ACE_inhibitor__base+classes intx__DBP_delta__INT_ACE_inhibitor__base+classes+pre intx__DBP_delta__INT_angiotensin_receptor_blocker__base+classes intx__DBP_delta__INT_angiotensin_receptor_blocker__base+classes+pre intx__DBP_delta__INT_beta_blocker__base+classes intx__DBP_delta__INT_beta_blocker__base+classes+pre intx__DBP_delta__INT_calcium_channel_blocker__base+classes intx__DBP_delta__INT_calcium_channel_blocker__base+classes+pre intx__DBP_delta__INT_diuretic__base+classes intx__DBP_delta__INT_diuretic__base+classes+pre intx__DBP_delta__INT_statin__base+classes intx__DBP_delta__INT_statin__base+classes+pre intx__DBP_logdelta__INT_ACE_inhibitor__base+classes intx__DBP_logdelta__INT_angiotensin_receptor_blocker__base+classes intx__DBP_logdelta__INT_beta_blocker__base+classes intx__DBP_logdelta__INT_calcium_channel_blocker__base+classes intx__DBP_logdelta__INT_diuretic__base+classes intx__DBP_logdelta__INT_statin__base+classes intx__DBP_prop__INT_ACE_inhibitor__base+classes intx__DBP_prop__INT_angiotensin_receptor_blocker__base+classes intx__DBP_prop__INT_beta_blocker__base+classes intx__DBP_prop__INT_calcium_channel_blocker__base+classes intx__DBP_prop__INT_diuretic__base+classes intx__DBP_prop__INT_statin__base+classes intx__LDL_delta__INT_ACE_inhibitor__base+classes intx__LDL_delta__INT_ACE_inhibitor__base+classes+pre intx__LDL_delta__INT_angiotensin_receptor_blocker__base+classes intx__LDL_delta__INT_angiotensin_receptor_blocker__base+classes+pre intx__LDL_delta__INT_beta_blocker__base+classes intx__LDL_delta__INT_beta_blocker__base+classes+pre intx__LDL_delta__INT_calcium_channel_blocker__base+classes intx__LDL_delta__INT_calcium_channel_blocker__base+classes+pre intx__LDL_delta__INT_diuretic__base+classes intx__LDL_delta__INT_diuretic__base+classes+pre intx__LDL_delta__INT_statin__base+classes intx__LDL_delta__INT_statin__base+classes+pre intx__LDL_logdelta__INT_ACE_inhibitor__base+classes intx__LDL_logdelta__INT_angiotensin_receptor_blocker__base+classes intx__LDL_logdelta__INT_beta_blocker__base+classes intx__LDL_logdelta__INT_calcium_channel_blocker__base+classes intx__LDL_logdelta__INT_diuretic__base+classes intx__LDL_logdelta__INT_statin__base+classes intx__LDL_prop__INT_ACE_inhibitor__base+classes intx__LDL_prop__INT_angiotensin_receptor_blocker__base+classes intx__LDL_prop__INT_beta_blocker__base+classes intx__LDL_prop__INT_calcium_channel_blocker__base+classes intx__LDL_prop__INT_diuretic__base+classes intx__LDL_prop__INT_statin__base+classes intx__SBP_delta__INT_ACE_inhibitor__base+classes intx__SBP_delta__INT_ACE_inhibitor__base+classes+pre intx__SBP_delta__INT_angiotensin_receptor_blocker__base+classes intx__SBP_delta__INT_angiotensin_receptor_blocker__base+classes+pre intx__SBP_delta__INT_beta_blocker__base+classes intx__SBP_delta__INT_beta_blocker__base+classes+pre intx__SBP_delta__INT_calcium_channel_blocker__base+classes intx__SBP_delta__INT_calcium_channel_blocker__base+classes+pre intx__SBP_delta__INT_diuretic__base+classes intx__SBP_delta__INT_diuretic__base+classes+pre intx__SBP_delta__INT_statin__base+classes intx__SBP_delta__INT_statin__base+classes+pre intx__SBP_logdelta__INT_ACE_inhibitor__base+classes intx__SBP_logdelta__INT_angiotensin_receptor_blocker__base+classes intx__SBP_logdelta__INT_beta_blocker__base+classes intx__SBP_logdelta__INT_calcium_channel_blocker__base+classes intx__SBP_logdelta__INT_diuretic__base+classes intx__SBP_logdelta__INT_statin__base+classes intx__SBP_prop__INT_ACE_inhibitor__base+classes intx__SBP_prop__INT_angiotensin_receptor_blocker__base+classes intx__SBP_prop__INT_beta_blocker__base+classes intx__SBP_prop__INT_calcium_channel_blocker__base+classes intx__SBP_prop__INT_diuretic__base+classes intx__SBP_prop__INT_statin__base+classes
"""

intx_runs = INTX_TEXT.split()

drug_of = lambda k: (k.rsplit("_", 1)[0] if k.rsplit("_", 1)[-1].isdigit() else k)

intx_by_key = {
    k: sorted(
        r for r in intx_runs
        if ("__SBP_" in r if "SBP" in k else
            "__DBP_" in r if "DBP" in k else
            (f"__INT_{drug_of(k)}__" in r and ("__SBP_" in r or "__DBP_" in r)))
    )
    for k in ci_to_gwas
}

for_int = cigwas[cigwas["phenotype"].isin(['SBP_post_ADJ','statin',
       'DBP_post_ADJ', 'SBP_post_1_ADJ', 'SBP_post_3_ADJ','angiotensin_receptor_blocker_1','statin_1',"DBP_post_3_ADJ"])]

def scan_pvals(path, ids, test):
    out = {}
    with path.open() as f:
        next(f)
        for ln in f:
            c = ln.split()
            vid = c[2]
            if vid in ids and c[7] == test and c[11] != "NA":
                out[vid] = 10 ** (-float(c[11]))
    return out

need = {}
for ph, g in for_int.groupby("phenotype"):
    runs = intx_by_key.get(ph, [])
    if not runs:
        continue
    for chr_, sub in g.groupby("chr"):
        ids = set(sub["rsID"].astype(str))
        for run in runs:
            need.setdefault((run, int(chr_)), set()).update(ids)

p_by = {}
for (run, chr_), ids in need.items():
    parts = run.split("__")
    trait = parts[1]
    drug  = parts[2].replace("INT_", "")
    test  = f"ADD-INT_SNPx{drug}"
    fpath = STEP2_ROOT / run / f"chr{chr_}" / f"{run}_chr{chr_}_step2_{trait}.regenie"
    for vid, p in scan_pvals(fpath, ids, test).items():
        p_by[(vid, run)] = p

def pvals_for_variant(phenotype, rsid, keep_missing=False):
    runs = intx_by_key.get(phenotype, [])
    if keep_missing:
        return {run: p_by.get((rsid, run), np.nan) for run in runs}

    out = {}
    for run in runs:
        p = p_by.get((rsid, run))
        if p is not None:
            out[run] = p
    return out


def best_from_pvals(d):
    best_run, best_p = None, np.inf
    for run, p in d.items():
        if p is None or (isinstance(p, float) and np.isnan(p)):
            continue
        if p < best_p:
            best_run, best_p = run, p
    return best_run, (best_p if best_run is not None else np.nan)

rsids = for_int["rsID"].astype(str).to_numpy()
phenos = for_int["phenotype"].to_numpy()

int_pvals = [pvals_for_variant(ph, vid, keep_missing=False) for ph, vid in zip(phenos, rsids)]

for_int = for_int.copy()
for_int["int_pvals"] = int_pvals

best = [best_from_pvals(d) for d in int_pvals]
for_int["int_lowest"] = [b[0] for b in best]
for_int["int_p"] = [b[1] for b in best]

for_int["int_pvals_json"] = for_int["int_pvals"].apply(lambda d: json.dumps(d, sort_keys=True))

for_int.to_csv("ci_gwas_interaction_lookup.tsv", sep = "\t")

### GWAS to ci lookup
# now I will do the lookup the other way. I have my clumps:
# I will read in all of the CI-GWAS results and look for any SNPs in ID or SP2 and collect their phenotypes and p-values

cl = clumps.reset_index(names="clump_row")

sp2 = (cl["SP2"].astype("string").fillna("").replace({".": ""}))
id_ = (cl["ID"].astype("string").fillna("").replace({".": ""}))

long = (
    cl.assign(rsID=(id_ + "," + sp2).str.split(","))
      .explode("rsID", ignore_index=True)[["clump_row", "rsID"]]
)

long["rsID"] = long["rsID"].astype("string").str.strip()
long = long.loc[long["rsID"].ne("")].drop_duplicates(["clump_row", "rsID"])

hits = (
    long.merge(cigwas[["rsID", "setup", "phenotype", "p_fdr"]], on="rsID", how="inner")
        .assign(key=lambda d: d["setup"].astype(str) + "::" + d["phenotype"].astype(str))
        .groupby(["clump_row", "key"], as_index=False)["p_fdr"].min()
)

out = clumps.copy()

ci_dict = hits.groupby("clump_row").apply(lambda g: dict(zip(g["key"], g["p_fdr"])))
out["ci_hits"] = out.index.to_series().map(ci_dict).apply(lambda x: x if isinstance(x, dict) else {})

out.to_csv("gwas-hits-in-ci.tsv", sep="\t")
